datosestacion = readtable("d684.csv");
%como todos los datos son numéricos
datos = table2array(datosestacion);

datos(datos == 9999) = NaN;

ano2023 = datos(15765:15794,1);
mesjunio = datos(15765:15794,2);
dias = datos(15765:15794,3);
registro = datos(15765:15794,4);

tiempo1 = datenum(ano2023,mesjunio,dias,0,0,0);

pf1 = polyfit(tiempo1,registro,6);
ajuste1 = polyval(pf1,tiempo1);

figure;
plot(tiempo1,registro,'-bx','LineWidth',2.5);

    title('Variación del nivel del mar (en mm) en la estación de Puerto Montt durante el mes de junio de 2023');
    xlabel('Fecha');
    ylabel('Nivel del mar (en mm)');
    datetick('x', 'dd-mmm','keeplimits');
    set(gca, 'XTickLabelRotation', 45);
    grid on;
    xlim([min(tiempo1) max(tiempo1)]);
    hold on;
    plot(tiempo1,ajuste1,'-xy','LineWidth',2);
    legend('Registro de la estación','Polinomio de grado 6 ajustado');


ndatosestacion = readtable("h684.csv");
%como todos los datos son numéricos
datos1 = table2array(ndatosestacion);

datos1(datos1 == 9999) = NaN;

nano2023 = datos1(378799:378822,1);
nmesjunio = datos1(378799:378822,2);
diasdelmes = datos1(378799:378822,3);
horasdeldia = datos1(378799:378822,4);
registros = datos1(378799:378822,5);

tiempo2 = datenum(nano2023,nmesjunio,diasdelmes,horasdeldia,0,0);

pf2 = polyfit(tiempo2,registros,1);
ajuste2 = polyval(pf2,tiempo2);

figure;
plot(tiempo2,registros,'-rx','LineWidth',2.5);

    title('Variación del nivel del mar (en mm) en la estación de Puerto Montt durante el 20 de junio de 2023');
    xlabel('Horas');
    ylabel('Nivel del mar (en mm)');
    datetick('x', 'HH:MM', 'keeplimits');
    set(gca, 'XTickLabelRotation', 45);
    grid on;
    hold on;
    plot(tiempo2,ajuste2,'-y','LineWidth',7);
    legend('Registro de la estación', 'Ajuste polinómico de grado 1');


figure;
subplot(2,1,1);
plot(tiempo1,registro,'-bx','LineWidth',2.5);

    title('Variación del nivel del mar (en mm) en la estación de Puerto Montt durante el mes de junio de 2023');
    xlabel('Fecha');
    ylabel('Nivel del mar (en mm)');
    datetick('x', 'dd-mmm','keeplimits');
    set(gca, 'XTickLabelRotation', 45);
    grid on;
    xlim([min(tiempo1) max(tiempo1)]);
    legend('Registro de la estación');

subplot(2,1,2);
plot(tiempo2,registros,'-rx','LineWidth',2.5);

    title('Variación del nivel del mar (en mm) en la estación de Puerto Montt durante el 20 de junio de 2023');
    xlabel('Horas');
    ylabel('Nivel del mar (en mm)');
    datetick('x', 'HH:MM', 'keeplimits');
    set(gca, 'XTickLabelRotation', 45);
    grid on;
    legend('Registro de la estación');

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from datetime import datetime

# --- PARTE 1: DATOS DIARIOS (Junio 2023) ---

# 1. Leer datos SIN CABECERA (header=None)
# Asignamos los nombres manualmente: Año, Mes, Día, Nivel
df = pd.read_csv('d684.csv', header=None, names=['Year', 'Month', 'Day', 'SeaLevel'])

# 2. Limpieza (Reemplazar 9999 por NaN)
# Aseguramos que sea numérico por si acaso
df['SeaLevel'] = pd.to_numeric(df['SeaLevel'], errors='coerce') 
df['SeaLevel'] = df['SeaLevel'].replace(9999, np.nan)

# 3. Crear columna de Fecha (DateTime)
df['Fecha'] = pd.to_datetime(df[['Year', 'Month', 'Day']])

# (El resto del código sigue igual...)
# 4. Filtrar datos
mask_junio = (df['Fecha'].dt.year == 2023) & (df['Fecha'].dt.month == 6)
datos_junio = df.loc[mask_junio].copy()

print(f"Leídos {len(df)} datos. Datos de junio encontrados: {len(datos_junio)}")
print(datos_junio.head()) # Para verificar que se ve bien

# 5. Ajuste Polinómico (Grado 6) - VERSIÓN CORREGIDA
fechas_num = mdates.date2num(datos_junio['Fecha'])
nivel_mar = datos_junio['SeaLevel']

# Validar que no haya NaNs
idx_validos = ~np.isnan(nivel_mar)

# --- EL TRUCO MATEMÁTICO: CENTRAR LOS DATOS ---
# Guardamos la fecha inicial como referencia (t=0)
fecha_referencia = fechas_num[idx_validos][0] 

# Creamos un vector de tiempo relativo (0, 1, 2 días...)
# Esto hace que los números sean pequeños y el polyfit no explote
tiempo_relativo = fechas_num[idx_validos] - fecha_referencia

# Hacemos el fit sobre el tiempo relativo (pequeño)
coef1 = np.polyfit(tiempo_relativo, nivel_mar[idx_validos], 6)
poly1 = np.poly1d(coef1)

# Para calcular la curva de ajuste, usamos también el tiempo relativo
tiempo_relativo_total = fechas_num - fecha_referencia
ajuste1 = poly1(tiempo_relativo_total)

# NOTA: Al graficar, sigues usando 'datos_junio['Fecha']' en el eje X
# para que se vean las fechas bonitas, pero el eje Y (ajuste1) 
# ya fue calculado matemáticamente bien.

# --- PARTE 2: DATOS HORARIOS (20 Junio 2023) ---
df2 = pd.read_csv('h684.csv', header=None, names=['Year', 'Month', 'Day', 'Hour', 'SeaLevel'])
df2['SeaLevel'] = df2['SeaLevel'].replace(9999, np.nan)
df2['Fecha'] = pd.to_datetime(df2[['Year', 'Month', 'Day', 'Hour']])

# Filtrar para el día 20 de Junio
mask_dia20 = (df2['Fecha'].dt.year == 2023) & (df2['Fecha'].dt.month == 6) & (df2['Fecha'].dt.day == 20)
datos_dia20 = df2.loc[mask_dia20].copy()

fechas_num2 = mdates.date2num(datos_dia20['Fecha'])
nivel_mar2 = datos_dia20['SeaLevel']

# Ajuste lineal (Grado 1)
coef2 = np.polyfit(fechas_num2, nivel_mar2, 1)
poly2 = np.poly1d(coef2)
ajuste2 = poly2(fechas_num2)

# ==========================================
# BLOQUE 3: GRAFICACIÓN
# ==========================================

# FIGURA 1: Junio Completo
plt.figure(figsize=(10, 6))
plt.plot(datos_junio['Fecha'], nivel_mar, '-bx', linewidth=2.5, label='Registro estación')
plt.plot(datos_junio['Fecha'], ajuste1, '-xy', linewidth=2, label='Polinomio grado 6')

plt.title('Variación nivel del mar - Puerto Montt (Junio 2023)')
plt.ylabel('Nivel del mar (mm)')
plt.xlabel('Fecha')
plt.grid(True)
plt.legend()

# Formato de fechas en eje X (El equivalente a datetick)
plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%d-%b'))
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# FIGURA 2: Día 20 Junio
plt.figure(figsize=(10, 6))
plt.plot(datos_dia20['Fecha'], nivel_mar2, '-rx', linewidth=2.5, label='Registro estación')
plt.plot(datos_dia20['Fecha'], ajuste2, '-y', linewidth=7, alpha=0.6, label='Ajuste lineal') # alpha para transparencia

plt.title('Variación nivel del mar - Puerto Montt (20 Junio 2023)')
plt.ylabel('Nivel del mar (mm)')
plt.xlabel('Hora')
plt.grid(True)
plt.legend()

# Formato HH:MM
plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# FIGURA 3: Subplots (2 en 1)
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 10))

# Subplot 1
ax1.plot(datos_junio['Fecha'], nivel_mar, '-bx', linewidth=2.5)
ax1.set_title('Mes de Junio 2023')
ax1.set_ylabel('Nivel (mm)')
ax1.grid(True)
ax1.xaxis.set_major_formatter(mdates.DateFormatter('%d-%b'))
ax1.tick_params(axis='x', rotation=45)

# Subplot 2
ax2.plot(datos_dia20['Fecha'], nivel_mar2, '-rx', linewidth=2.5)
ax2.set_title('Día 20 de Junio')
ax2.set_ylabel('Nivel (mm)')
ax2.set_xlabel('Horas')
ax2.grid(True)
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
ax2.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

df : DataFrame o los datos crudos leídos
pd.read_csv : Usa pandas para leer el archivo, se especifica el nombre del archivo, que no tiene una primera fila con nombres de variables, y se les asigna los nombres de cada columna manualmente
df['SeaLevel'] = df['SeaLevel'].replace(9999, np.nan) : Aislamos la columna SeaLevel y se reemplaza los datos 9999 por NaNs
df['Fecha'] = pd.to_datetime(df[['Year', 'Month', 'Day']]) : Creamos una nueva variable Fecha, usando pandas para obtener el datetime que es una columna con Year, Month, Day extraídos del csv
mask_junio = (df['Fecha'].dt.year == 2023) & (df['Fecha'].dt.month == 6) _ Extraímos solo los datos del año 2023 y los datos que corresponden al mes 6, así obtenemos los datos de Junio 2023
datos_junio = df.loc[mask_junio].copy() : Localizamos los datos obtenidos y los copiamos para así obtener la variable que contiene todos los datos de Junio 2023
fechas_num = mdates.date2num(datos_junio['Fecha']) : Crea una variable que tiene las fechas convertidas en números en orden para poder graficarlas en el eje X, date2num es el datenum de Matlab
nivel_mar = datos_junio['SeaLevel'] : Crea una variable con todos los datos de nivel del mar de junio 2023
idx_validos = ~np.isnan(nivel_mar) : Crea otra variable con los datos del nivel del mar recolectados quitando los NaNs
fecha_referencia = fechas_num[idx_validos][0] : Crea una variable escalar que es el primer día o el primer dato que se tiene
tiempo_relativo = fechas_num[idx_validos] - fecha_referencia : Crea una variable que es una lista de todos los datos válidos y se le resta el inicial, de esa forma se mueve el inicio del eje X
coef1 = np.polyfit(tiempo_relativo, nivel_mar[idx_validos], 6) : Polyfit de Matlab, crea los datos del eje X a través de las variables X e Y
poly1 = np.poly1d(coef1) : Crea y guarda la función



